# HPC Cluster Concepts

## Cluster Overview

<img src="./img/general_cluster_architecture.png"/>

A **cluster** is a collection of compute nodes. A **partition** is a logical grouping of nodes or resources. It is used to organize access to different types of hardware, such as CPU nodes, GPU nodes, or nodes with special characteristics.

A **node** is one compute server. Each node can contain one or more CPU sockets. A socket is the physical CPU package installed in the server.

Each socket contains multiple **CPU cores**. A CPU core is a physical computing unit. Depending on the processor and configuration, each core may expose one or more **CPU hardware threads**, typically two when simultaneous multithreading or hyperthreading is enabled. If this is disabled, one hardware thread usually corresponds to one physical core.

Modern processors are often divided into **NUMA domains**, also called locality domains. A NUMA domain is a group of CPUs that share memory that is physically closer to them. Accessing local memory is faster than accessing memory attached to another NUMA domain. A single socket can therefore contain one or more NUMA domains, depending on the processor architecture and BIOS configuration.

Some nodes may also contain **GPUs**, which are accelerator processors. GPUs can have locality relationships with specific CPU sockets or NUMA domains, so their placement in the hardware topology can matter for performance.

A **task** is a running process, for example an *MPI rank*. **Affinity** describes where that process is allowed to run: on which CPU, core, socket, or NUMA domain. Correct affinity helps keep processes close to the CPU cores, memory, and accelerators they use.

A simplified hierarchy is:

```shell
Cluster
  └── Partition
       └── Node
            └── Socket
                 └── NUMA / Locality Domain
                      └── CPU Core
                           └── CPU Hardware Thread
```

The key message is that performance is not only about how many CPUs a system has, but also about **where those CPUs, memory, tasks, and accelerators are located relative to each other.**

## Slurm Cluster Overview

<img src="./img/general_slurm_architecture.png"/>

**Slurm** is the workload management layer that connects users to the compute infrastructure. In other words, **the HPC cluster provides the resources, while Slurm manages access to them**. It queues jobs, allocates resources, launches workloads on compute nodes, and records usage for accounting and reporting.

An **HPC cluster** provides the physical and shared resources: login nodes, compute nodes, CPUs, memory, GPUs, high-speed network, and often **shared storage**. Users normally access the cluster through login nodes or other entry points, but heavy workloads should not run directly there. Instead, users submit jobs to Slurm using commands such as `sbatch` or `srun`.

Slurm receives these job requests, places them in a queue, and decides when and where they can run based on available resources, partitions, priorities, limits, and site policies. The Slurm controller, **slurmctld**, is responsible for scheduling and launching jobs on the selected compute nodes.

Once resources are available, Slurm starts the job on the compute nodes allocated to it. These nodes provide the CPUs, memory, and GPUs requested by the job.

Slurm also optionally includes an accounting component called **slurmdbd**. The **slurmdbd** service collects and stores job accounting information in a database. This includes details such as who ran a job, when it started and ended, which resources were allocated, how long the job ran, and how much CPU, memory, or GPU time was consumed. This information is used for reporting, fair-share calculations, usage analysis, debugging, and long-term accounting.

### The General HPC Slurm cluster

At **Paul Scherrer Institut**, **Merlin** is a series of centrally managed high-performance computing clusters, and is providing PSI staff and collaborators with scalable CPU and GPU resources for simulations, data analysis, and other data-intensive scientific workloads. This is operated by the [HPCE team](https://www.psi.ch/en/awi/hpce-group), in the [Scientific Computing Division](https://www.psi.ch/en/csd). PSI also run other Slurm clusters

**Merlin7** is the newest generation of the Merlin HPC environment. The cluster is based on Slurm, and is hosted on CSCS's Alps infrastructure in Lugano as an independent vCluster, and provides PSI users with flexible access to CPU and modern GPU resources for demanding scientific computing workloads.

In addition to Merlin, PSI also operates more specialized computing clusters for specific facilities, projects, or departments, such as the **Ra** data analysis cluster and the **SwissFEL** online/near-time computing infrastructure, which are tailored to experimental data acquisition, processing, and analysis workflows.

<img src="./img/merlin7_architecture.png"/>

## Merlin7 Slurm architecture overview

Merlin7 is composed of **two Slurm clusters** that share the same accounting layer and are accessed through login or service entry points.

Users can access Merlin7 and submit jobs through several **entry points**:

- **`login001.merlin7.psi.ch` / `login002.merlin7.psi.ch`** are the interactive login nodes used by users to prepare and submit jobs. Access is possible through SSH protocol as well as NoMachine (desktop based). Typical commands for submitting jobs from the login nodes are: `sbatch`, `salloc`, `srun`.
- **`merlin7-jupyter.psi.ch` / `ondemand.psi.ch`** are web-based entry points which provide browser-based access to cluster services.
- **`service03`** is a non-interactive service node, used for application-driven job submissions

Merlin7 has two separate Slurm control planes:

* The **`merlin7`** Slurm cluster manages the CPU resources. Is managed by **2 `slurmctld` controllers** in a high-availability setup (**active/passive**) which are in charge of scheduling jobs to the Merlin7 CPU nodes
* The **`gmerlin7`** Slurm cluster manages the GPU resources. Is managed by **2 `slurmctld` controllers** in a high-availability setup (**active/passive**) which are in charge of scheduling jobs to the Merlin7 GPU nodes

Both Slurm clusters share the same accounting infrastructure. This layer consists of **2 `slurmdbd` services** in a high-availability setup (**active/passive**), responsible for collecting and exposing accounting information from both Slurm clusters, as well as managing Slurm users, accounts, and associations. The `slurmdbd` services are backed by a **MariaDB** database, which stores user and account information, job history, resource usage, and accounting records. This shared accounting layer allows both `merlin7` and `gmerlin7` to report usage consistently through the same backend.

The compute resources are divided into CPU and GPU systems:

* Merlin7 CPU nodes are AMD-based compute nodes and are exclusively used for CPU workloads. These nodes are managed by the `merlin7` Slurm cluster.
* Merlin7 GPU nodes are a mix of A100 (AMD-based) and Grace-Hopper (ARM-based) nodes, which are exclusively used for GPU workloads. These nodes are managed by the `gmerlin7` Slurm cluster.

The HPC system (login and compute nodes, and storage) is connected through the **Slingshot interconnect / system network backbone**. 

The shared storage layer is provided by and **HPC Lustre Storage**, which is a shared parallel filesystem accessible from the cluster components. This storage is used for user data, project data and shared scratch.

## Hands-on: Accessing the Merlin7 cluster

**Duration:** ~15 minutes.

### Exercise 1: Log in using your preferred method

Access to the Merlin7 login nodes using your preferred protocol. For this, please use the PSI username (`psicourse-stud[01-50]`) and password which was provided to you. Possible options:

* Using SSH, as described [here](https://hpce.pages.psi.ch/merlin7/01-Quick-Start-Guide/accessing-interactive-nodes/#ssh-access)
* Using NoMachine, as described [here](https://hpce.pages.psi.ch/merlin7/02-How-To-Use-Merlin/nomachine/)
* Using Open OnDemand, as described [here](https://hpce.pages.psi.ch/merlin7/05-Open-OnDemand/open-ondemand/)


## Hands-on: understanding hardware topology, NUMA, cores, threads, and affinity

### Duration
Around 15 minutes.

#### Exercise 2: Identify the machine CPU and NUMA topology

Understand the basic hardware hierarchy of the nodes.

On the login node, run:

```bash
lscpu
```

Then focus on the most relevant fields:

```bash
lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'
```

Now run the same command on a CPU compute node, as follows:

```bash
srun -p interactive --reservation interactive \
  bash -lc "lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'"
```

Finally, run the same command on a GPU compute node, as follows:

```bash
srun -M gmerlin7 -p a100-hourly --reservation psicourse01 \
  bash -lc "lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'"
```

##### Questions

> **[INFO]** Edit this cell and write your answers below.

1. **How many sockets do the nodes have?**
> YOUR ANSWER HERE

3. **How many physical cores per socket?**
> YOUR ANSWER HERE

4. **How many hardware threads per core?**
> YOUR ANSWER HERE

5. **How many total hardware threads?**
> YOUR ANSWER HERE

6. **How many NUMA domains? Why do the CPU login and compute nodes differ if the processor is the same?**
> YOUR ANSWER HERE


<img src="./img/slurm_basic.png"/>

## Basic Slurm command workflow

We can divide common Slurm commands into four parts of the job lifecycle:
1. submitting or launching work,
2. monitoring the queue,
3. controlling jobs,
4. and checking accounting or efficiency information.

The typical Slurm workflow starts with submitting or launching work through `sbatch`, `srun`, or `salloc`. Users then monitor their jobs and the cluster state with `squeue`, `sinfo`, and `sprio`.

If needed, jobs can be cancelled or inspected in more detail with `scancel` and `scontrol`. After completion, `sacct`, `sacctmgr`, and `seff` help users understand accounting information, limits, usage, and job efficiency.

### Submit and launch jobs

Users normally submit work to Slurm instead of running heavy workloads directly on login nodes. For batch workloads, `sbatch` is used to submit a job script to the scheduler. This is the standard approach for production jobs, longer calculations, and workflows that do not require direct interaction.

For interactive work, users can use `srun` or `salloc`. These commands request resources from Slurm and provide an interactive environment once the allocation is available. This is useful for testing, debugging, compiling, short exploratory runs, or preparing job scripts before submitting larger batch jobs.

The important idea is that users request compute resources from Slurm, and Slurm decides where and when the workload can run.

### Monitor jobs and cluster state

After submitting a job, users can inspect its status with `squeue`. This shows whether jobs are running, pending, or waiting for resources or policy conditions.

The `sinfo` command gives an overview of partitions and node states. It helps users understand which parts of the cluster are available and whether nodes are idle, allocated, drained, or unavailable.

The `sprio` command provides information about job priority. It can help explain why a pending job is behind another job in the queue, especially when fair-share, job age, partition, or other scheduling factors are involved.

Together, these commands help users answer basic questions such as whether their job is pending, which partitions are available, and why another job may start earlier.

### Control jobs

Users can cancel jobs with `scancel`. This is useful when a job was submitted with wrong parameters, is no longer needed, or is not behaving as expected.

For more detailed inspection and control, `scontrol` provides access to Slurm’s internal view of jobs, nodes, and partitions. It can be used to inspect detailed job information, hold or release jobs, and, for administrators, modify selected job or cluster parameters.

Some `scontrol` actions are available to normal users for their own jobs, while others are restricted to administrators.

### Account and tune jobs

After jobs have run, users can inspect accounting information with `sacct`. This provides job history and resource usage information, including completed, failed, cancelled, or running jobs, depending on site configuration.

The `sacctmgr` command is used for Slurm accounting management. It exposes information about accounts, users, associations, fair-share, QoS, and limits. This is especially useful for administrators, but it can also help explain scheduling behavior and resource limits.

For completed batch jobs, `seff` gives a simple efficiency summary. It helps users understand whether the requested CPUs, memory, and walltime matched the actual job usage. This information is useful for tuning future submissions and improving cluster efficiency.


## Hands-on: submitting, monitoring, controlling, and tuning Slurm jobs

This hands-on introduces the basic Slurm workflow from the user point of view. The goal is not only to submit a job, but also to understand what resources are being requested and how to inspect what Slurm did with the job afterwards.

The exercises are intentionally small and safe. They use commands such as hostname, sleep, and short shell or Python snippets, so they should not create significant load on the system.

### Duration
Around 40 minutes.

#### Exercise 1: Understand the resource request

Before submitting jobs, it is important to understand the most common Slurm resource options. A Slurm job request describes the *shape* of the resources needed by the application: how many nodes, how many tasks, how many CPU cores, how much memory, how much time, and, when relevant, how many GPUs.

On Merlin7, users should also be aware that there are two Slurm clusters:

- `merlin7` for CPU workloads
- `gmerlin7` for GPU workloads

The target Slurm cluster can be selected with the `--clusters` option, or with the equivalent short option `-M`.

<table>
  <thead>
    <tr>
      <th>Option</th>
      <th>Meaning</th>
      <th>Typical use</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>--clusters</code> / <code>-M</code></td>
      <td>Selects the Slurm cluster to which the command or job applies.</td>
      <td>
        Use <code>--clusters=merlin7</code> for CPU jobs and
        <code>--clusters=gmerlin7</code> for GPU jobs.
      </td>
    </tr>
    <tr>
      <td><code>--partition</code></td>
      <td>Selects the partition where the job should run.</td>
      <td>
        Used to select a class of resources or policy, for example CPU, GPU,
        short, long, interactive, or project-specific partitions.
      </td>
    </tr>
    <tr>
      <td><code>--reservation</code></td>
      <td>Requests nodes from an existing Slurm reservation.</td>
      <td>
        Can be used on some systems to select an active reservation which
        contains reserved resources on specific nodes. Reservations are
        often created by administrators or users with granted privileges.
      </td>
    </tr>
    <tr>
      <td><code>--nodes</code></td>
      <td>Number of compute nodes requested.</td>
      <td>
        Use when the job must run on one or more physical nodes. Many beginner
        jobs should start with <code>--nodes=1</code>.
      </td>
    </tr>
    <tr>
      <td><code>--ntasks</code></td>
      <td>Number of Slurm tasks to start.</td>
      <td>
        Often used for MPI ranks or independent parallel processes.
      </td>
    </tr>
    <tr>
      <td><code>--ntasks-per-node</code></td>
      <td>Controls how many tasks are placed on each node.</td>
      <td>
        Useful for multi-node jobs where the task distribution across nodes
        matters.
      </td>
    </tr>
    <tr>
      <td><code>--cpus-per-task</code></td>
      <td>Number of CPU cores assigned to each task.</td>
      <td>
        Used for threaded applications, OpenMP, Python multiprocessing, or any
        program where one process can use multiple CPU cores.
      </td>
    </tr>
    <tr>
      <td><code>--mem</code></td>
      <td>Memory requested per node.</td>
      <td>
        Use when the job needs a fixed amount of memory for the node allocation.
      </td>
    </tr>
    <tr>
      <td><code>--mem-per-cpu</code></td>
      <td>Memory requested per allocated CPU.</td>
      <td>
        Useful when the memory request should scale with the number of allocated
        CPU cores.
      </td>
    </tr>
    <tr>
      <td><code>--mem-per-gpu</code></td>
      <td>Memory requested per allocated GPU.</td>
      <td>
        Useful for GPU jobs when the memory request should scale with the number
        of GPUs. Do not combine it with <code>--mem</code> or
        <code>--mem-per-cpu</code>.
      </td>
    </tr>
    <tr>
      <td><code>--gpus</code></td>
      <td>Total number of GPUs requested for the job.</td>
      <td>
        Simple GPU request, typically used on <code>gmerlin7</code>.
      </td>
    </tr>
    <tr>
      <td><code>--gpus-per-node</code></td>
      <td>Number of GPUs requested on each allocated node.</td>
      <td>
        Useful when the job spans multiple GPU nodes and needs a fixed number
        of GPUs per node.
      </td>
    </tr>
    <tr>
      <td><code>--gpus-per-task</code></td>
      <td>Number of GPUs assigned per Slurm task.</td>
      <td>
        Useful when each task should receive one or more GPUs, for example in
        multi-process GPU workloads.
      </td>
    </tr>
    <tr>
      <td><code>--cpus-per-gpu</code></td>
      <td>Number of CPU cores requested per allocated GPU.</td>
      <td>
        Useful for GPU jobs that need CPU cores to feed each GPU efficiently.
      </td>
    </tr>
    <tr>
      <td><code>--gres=gpu:&lt;N&gt;</code></td>
      <td>Requests GPUs through Slurm generic resources.</td>
      <td>
        Common on many Slurm systems. Depending on the site configuration,
        this may be used instead of, or alongside, <code>--gpus</code>.
      </td>
    </tr>
    <tr>
      <td><code>--time</code></td>
      <td>Maximum runtime of the job.</td>
      <td>
        Helps Slurm schedule the job and defines when the job will be stopped
        if it exceeds its limit.
      </td>
    </tr>
  </tbody>
</table>

A common source of confusion is the difference between tasks and CPUs per task:

* A job with `--ntasks=4 --cpus-per-task=1` means: *start four separate tasks, each with one CPUs.* This is typical for MPI-style jobs or independent parallel processes.
* A job with `--ntasks=1 --cpus-per-task=4` means *start one task, but give it four CPUs.* This is typical for threaded applications. The application must actually use those threads; otherwise the extra CPUs are allocated but mostly idle.

For GPU jobs, the same idea still applies: CPU tasks and GPU resources are related, but they are not the same thing. For example, a job may request one task, several CPU threads, and one GPU: `--clusters=gmerlin7 --ntasks=1 --cpus-per-task=8 --gpus=1`. This means: *run one task, give it eight CPUs, and allocate one GPU.* This is a common shape for a single-process GPU application.

Memory options should be chosen carefully. In general, use only one of: `--mem`, `--mem-per-cpu` or `--mem-per-gpu`:
* Use `--mem` when the job needs a fixed amount of memory per node.
* Use `--mem-per-cpu` when memory should scale with the number of CPU cores. On Merlin7, we have a default of `2912M` for the GPU cluster and `1888MB` for the CPU cluster
* Use `--mem-per-gpu` when memory should scale with the number of GPUs.

The main idea is that a Slurm resource request should describe what the application can really use. Requesting more CPUs, memory, nodes, or GPUs does not automatically make a job faster. It may only make the job harder to schedule if the application cannot use those resources efficiently.

## Exercise 2: Inspect the cluster before submitting jobs

Start by checking where you are:

```bash
hostname
whoami
pwd
```

You should be on a login node. Heavy computations should not run here directly; they should be submitted to Slurm.

Now inspect the available partitions:

```bash
sinfo
echo "\"$SINFO_FORMAT\""
```

For a different detail:

```bash
SINFO_FORMAT="%.16P %.14F %.14C %.16L %.14l %.40G %.5D %N" sinfo --clusters=all
```

This shows partition names, availability, time limits, node counts, and node states. It helps answer the question: *where can my job run?*

Also check whether you already have jobs in the queue, as well as running and pending jobs on the GPU cluster:

```bash
squeue -u $USER
squeue -M gmerlin7 -t R
squeue -M gmerlin7 -t PD
echo "\"$SQUEUE_FORMAT\""
```

At this point, the important concept is that Slurm sees the cluster as a set of partitions and nodes. Users submit jobs to partitions, and Slurm decides when and where the job can run.

### Exercice 3: Submit a first serial batch job

Create a simple batch script:

```bash
cat > 01_serial.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=ho-serial
#SBATCH --output=logs/%x-%j.out
#SBATCH --time=00:02:00
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=1G

set -euo pipefail

echo "Job ID:        $SLURM_JOB_ID"
echo "Job name:      $SLURM_JOB_NAME"
echo "Running on:    $(hostname)"
echo "Submit dir:    $SLURM_SUBMIT_DIR"
echo "Nodes:         $SLURM_JOB_NUM_NODES"
echo "Tasks:         $SLURM_NTASKS"
echo "CPUs per task: $SLURM_CPUS_PER_TASK"

echo
echo "Doing a small test..."
sleep 60
echo "Done."
EOF
```

Submit it:

```bash
JOBID=$(sbatch --parsable --partition="$PARTITION" 01_serial.slurm)
echo "Submitted job: $JOBID"
```

Monitor it:

```bash
squeue -j "$JOBID"
```

Inspect Slurm’s detailed view of the job:

```bash
scontrol show job "$JOBID"
```

Once the job starts, look at the output file:

```bash
cat logs/ho-serial-${JOBID}.out
```

This first job requested one node, one task, one CPU per task, and 1 GB of memory. This is the simplest possible pattern for a serial job.

### Exercise 4: Compare tasks and CPUs per task

Now create a job with multiple tasks. This job asks Slurm to launch four separate tasks on one node.

```bash
cat > 02_multi_task.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=ho-tasks
#SBATCH --output=logs/%x-%j.out
#SBATCH --time=00:05:00
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=500M

set -euo pipefail

echo "Job ID:        $SLURM_JOB_ID"
echo "Nodes:         $SLURM_JOB_NUM_NODES"
echo "Tasks:         $SLURM_NTASKS"
echo "CPUs per task: $SLURM_CPUS_PER_TASK"

echo
echo "Launching one command per Slurm task:"
srun bash -c 'echo "task=$SLURM_PROCID local_task=$SLURM_LOCALID node=$(hostname) cpus_per_task=$SLURM_CPUS_PER_TASK"; sleep 20'
EOF
```

Submit and monitor it:

```bash
JOBID_TASKS=$(sbatch --parsable --partition="$PARTITION" 02_multi_task.slurm)
echo "Submitted job: $JOBID_TASKS"

squeue -j "$JOBID_TASKS"
```

After completion:

```bash
cat logs/ho-tasks-${JOBID_TASKS}.out
```

You should see four task lines. The important point is that `--ntasks=4` means Slurm can start four task instances. This is the model used by MPI jobs and by workflows that run several independent processes.

Now compare that with a threaded-style job:

```bash
cat > 03_threaded_style.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=ho-threaded
#SBATCH --output=logs/%x-%j.out
#SBATCH --time=00:05:00
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=2G

set -euo pipefail

export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK

echo "Job ID:        $SLURM_JOB_ID"
echo "Running on:    $(hostname)"
echo "Tasks:         $SLURM_NTASKS"
echo "CPUs per task: $SLURM_CPUS_PER_TASK"
echo "OMP threads:   $OMP_NUM_THREADS"

echo
echo "This allocation is suitable for one process that can use multiple CPU threads."
sleep 30
EOF
```

Submit it:

```bash
JOBID_THREADED=$(sbatch --parsable --partition="$PARTITION" 03_threaded_style.slurm)
echo "Submitted job: $JOBID_THREADED"
```

Inspect the output:

```bash
cat logs/ho-threaded-${JOBID_THREADED}.out
```

This job starts only one Slurm task, but that task receives four CPU cores. This is only useful if the application is able to use multiple threads. For OpenMP applications, setting `OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK` is a common pattern.